# RAG retrieval experiments — first end-to-end test of the paragraph layer

**Question:** does paragraph-granularity retrieval actually answer real questions about the
corpus, and which retrieval shape works best? Runs on storage v2 (`hrono-v2.db`) with the
experiment model (granite-97m); vectors in the `(hash, model)`-keyed `embeddings` table.

**Approaches compared, per prompt** (paragraph-injection only — chunk injection is out of
scope by design: chunks are the traversal unit, paragraphs the injection unit):
1. `top-k paragraphs` — plain cosine over paragraph vectors; the baseline injection set.
2. `MMR paragraphs` — maximal marginal relevance (λ=0.5, pool 30): trades raw similarity
   for diversity, against near-duplicate hits (translation files, repeated imports).
3. `hybrid RRF` — two legs with opposite strengths: the vector leg (meaning) + a BM25
   keyword leg (exact tokens; SQLite FTS5 over path + body, query terms OR-ed),
   merged by reciprocal rank fusion: score = Σ 1/(60 + rank), so agreement between legs
   wins and either leg alone can still carry a hit the other missed. Hits found only by
   the keyword leg are marked `<- keyword-only`.

Every hit prints as its one-line injection header (uri + `sp:` struct paths, sha shortened)
+ score + the FULL paragraph body — exactly what would enter the agent's context.
Observations at the end.

In [ ]:
import json
import re
import sqlite3
import time
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

APP_DIR = Path.cwd()
DB_PATH = APP_DIR / ".hrono" / "hrono-v2.db"
GIT_CACHE = APP_DIR / ".hrono" / "cache" / "git"
MODEL_NAME = "ibm-granite/granite-embedding-97m-multilingual-r2"
BATCH = 16

db = sqlite3.connect(DB_PATH)
db.execute("""
CREATE TABLE IF NOT EXISTS embeddings (
    hash   TEXT    NOT NULL,  -- paragraphs.hash or chunks.hash — content-addressed
    model  TEXT    NOT NULL,
    dims   INTEGER NOT NULL,
    vector BLOB    NOT NULL,  -- float32 LE, unit-normalized
    PRIMARY KEY (hash, model)
)""")

t0 = time.monotonic()
model = SentenceTransformer(MODEL_NAME)
print(f"model load: {time.monotonic() - t0:.1f}s")


def embed_missing(texts_by_hash: dict) -> None:
    done = {r[0] for r in db.execute(
        "SELECT hash FROM embeddings WHERE model = ?", (MODEL_NAME,))}
    todo = [h for h in texts_by_hash if h not in done]
    print(f"{len(texts_by_hash)} unique hashes, {len(done)} already embedded, {len(todo)} to embed")
    t0 = time.monotonic()
    for i in range(0, len(todo), BATCH):
        batch = todo[i:i + BATCH]
        vecs = model.encode([texts_by_hash[h] for h in batch], batch_size=BATCH,
                            normalize_embeddings=True, show_progress_bar=False)
        with db:
            db.executemany(
                "INSERT INTO embeddings (hash, model, dims, vector) VALUES (?,?,?,?)",
                [(h, MODEL_NAME, vecs.shape[1], np.asarray(v, np.float32).tobytes())
                 for h, v in zip(batch, vecs)])
    if todo:
        el = time.monotonic() - t0
        print(f"embedded {len(todo)} in {el:.0f}s — {1000 * el / len(todo):.0f} ms/text")


# paragraphs: bodies from contents (content-addressed, one embed per unique body)
para_texts = {h: b.decode("utf-8", "replace")
              for h, b in db.execute("SELECT hash, body FROM contents")}
embed_missing(para_texts)


In [ ]:
from collections import defaultdict
from urllib.parse import unquote


def load_level(table):
    rows = db.execute(f"""
        SELECT t.hash, t.uri, t.struct_uris, e.vector FROM {table} t
        JOIN embeddings e ON e.hash = t.hash AND e.model = ?
        GROUP BY t.hash""", (MODEL_NAME,)).fetchall()
    hashes = [r[0] for r in rows]
    headers = [f"{r[1]} {r[2]}".strip() for r in rows]
    mat = np.frombuffer(b"".join(r[3] for r in rows), np.float32).reshape(len(rows), -1)
    return hashes, headers, mat


p_hashes, p_headers, P = load_level("paragraphs")
header_by_hash = dict(zip(p_hashes, p_headers))
print(f"{len(p_hashes)} paragraph vectors")

# BM25 leg: FTS5 over path + body only — struct paths excluded (same argument as
# for embeddings: names already live in the bodies, the rest is grammar noise)
db.executescript("DROP TABLE IF EXISTS para_fts; CREATE VIRTUAL TABLE para_fts USING fts5(text);")
with db:
    db.executemany(
        "INSERT INTO para_fts (rowid, text) VALUES (?,?)",
        [(pid, f"{path} {body.decode('utf-8', 'replace')}")
         for pid, path, body in db.execute(
            "SELECT p.id, p.path, c.body FROM paragraphs p"
            " JOIN contents c ON c.hash = p.hash")])
hash_by_id = dict(db.execute("SELECT id, hash FROM paragraphs"))
print(f"FTS5 index: {db.execute('SELECT COUNT(*) FROM para_fts').fetchone()[0]} rows")


def top_k_paragraphs(q, prompt, k=5):
    sims = P @ q
    return [(sims[i], p_headers[i], p_hashes[i]) for i in np.argsort(-sims)[:k]]


def mmr_paragraphs(q, prompt, k=5, pool=30, lam=0.5):
    sims = P @ q
    cand = list(np.argsort(-sims)[:pool])
    picked = [cand.pop(0)]
    while cand and len(picked) < k:
        pen = np.max(P[cand] @ P[picked].T, axis=1)   # similarity to already-picked
        scores = lam * sims[cand] - (1 - lam) * pen
        picked.append(cand.pop(int(np.argmax(scores))))
    return [(sims[i], p_headers[i], p_hashes[i]) for i in picked]


def bm25_hashes(prompt, n=60):
    """Top-n unique hashes by BM25. Natural-language prompt -> OR-ed word terms
    (FTS5 defaults to AND, far too strict for questions)."""
    terms = re.findall(r"[A-Za-z0-9_]+", prompt)
    rows = db.execute(
        "SELECT rowid FROM para_fts WHERE para_fts MATCH ? ORDER BY bm25(para_fts) LIMIT ?",
        (" OR ".join(terms), n * 3)).fetchall()
    ranked = list(dict.fromkeys(hash_by_id[rid] for (rid,) in rows))
    return ranked[:n]


def hybrid_rrf(q, prompt, k=5, K=60):
    sims = P @ q
    vec_ranked = [p_hashes[i] for i in np.argsort(-sims)[:K]]
    kw_ranked = bm25_hashes(prompt, K)
    score = defaultdict(float)
    for ranked in (vec_ranked, kw_ranked):
        for r, h in enumerate(ranked):
            score[h] += 1.0 / (K + r + 1)
    top = sorted(score, key=score.get, reverse=True)[:k]
    return [(score[h], header_by_hash[h] + ("" if h in vec_ranked else "   <- keyword-only"), h)
            for h in top]


APPROACHES = [("top-k paragraphs", top_k_paragraphs), ("MMR paragraphs", mmr_paragraphs),
              ("hybrid RRF (vector + BM25)", hybrid_rrf)]

## The prompts

Realistic questions with known answer locations spread across file kinds — code, config,
docs, data — including the motivating case for the whole paragraph layer: a question whose
answer is one small identifier deep in one file.

In [ ]:
PROMPTS = [
    "What does ContactService.getByUser return and how is pagination handled?",
    "Which Spring Boot starter dependencies does the football match project use?",
    "How do I run the football match application locally?",
    "How does the webphone client translate its UI into Ukrainian?",
    "Under which license is webphone distributed?",
    "Where is the photo of a contact validated for maximum size?",
    "How does the webphone client register a SIP account and connect over WebSocket?",
    "Which database does the webphone server use to store contacts?",
    "What Java version is required to build the webphone server?",
    "How are call history records fetched and ordered for a user?",
    # no-answer controls: none of these exist anywhere in the corpus
    "How is payment processing implemented and which payment provider API is used?",
    "Where is the Kafka topic configuration and how are events consumed?",
    "Which Helm chart values configure the Kubernetes ingress?",
]

body_of = dict(db.execute("SELECT hash, body FROM contents"))

for prompt in PROMPTS:
    q = model.encode([prompt], normalize_embeddings=True)[0]
    print("#" * 110)
    print(f"PROMPT: {prompt}")
    for name, fn in APPROACHES:
        print(f"\n=== {name} " + "=" * (100 - len(name)))
        for score, header, h in fn(q, prompt):
            print("-" * 110)
            print(f"{score:.3f}  {re.sub(r'@[0-9a-f]{40}', lambda m: m.group(0)[:9], header)}")
            print(body_of[h].decode("utf-8", "replace"))
    print()

## Path-aware retrieval — the clustering α-blend, applied to search

Same trick as clustering: paragraph vector = `[√(1-α)·content ; √α·path]`, query embedded
as `[√(1-α)·q ; √α·q]`, so the dot product is `(1-α)·sim(query,content) + α·sim(query,path)`
with no pipeline change. Struct paths deliberately NOT embedded: grammar segments are
noise and the named parts already live in the body text. Sweep α ∈ {0, 0.3, 0.5} over the
same prompts — watch the two path-shaped failures (Ukrainian, license) and whether the
clean wins survive the blend.

In [ ]:
path_by_hash = dict(db.execute("SELECT hash, path FROM paragraphs GROUP BY hash"))
distinct_paths = sorted(set(path_by_hash.values()))
pv = model.encode(distinct_paths, batch_size=BATCH, normalize_embeddings=True,
                  show_progress_bar=False)
path_vec = dict(zip(distinct_paths, pv))
PATH = np.stack([path_vec[path_by_hash[h]] for h in p_hashes])
print(f"embedded {len(distinct_paths)} distinct paths")

ALPHAS = (0.0, 0.3, 0.5)
BLENDS = {a: np.hstack([np.sqrt(1 - a) * P, np.sqrt(a) * PATH]) for a in ALPHAS}


def short(header):
    return re.sub(r'@[0-9a-f]{40}', lambda m: m.group(0)[:9], header)


def snip(h, n=160):
    text = " ".join(body_of[h].decode("utf-8", "replace").split())
    return text[:n] + ("…" if len(text) > n else "")


for prompt in PROMPTS:
    q = model.encode([prompt], normalize_embeddings=True)[0]
    print("#" * 110)
    print(f"PROMPT: {prompt}")
    for a, M in BLENDS.items():
        qa = np.concatenate([np.sqrt(1 - a) * q, np.sqrt(a) * q])
        sims = M @ qa
        print(f"\n--- alpha={a} ---")
        for i in np.argsort(-sims)[:3]:
            print(f"  {sims[i]:.3f}  {short(p_headers[i])}")
            print(f"         {snip(p_hashes[i])}")
    print()

## Adaptive k — how many hits deserve injection (possibly zero)

Fixed top-k is wrong in both directions: it injects garbage when nothing matches and
truncates when many things match equally well. Mechanism, per query:
1. **Self-calibrating significance floor** — score ALL paragraphs; the mean μ and std σ of
   that distribution are this query's background noise. A candidate qualifies only with
   z = (s−μ)/σ ≥ Z_MIN — "far above background", robust to model/corpus shifts where an
   absolute cosine floor is not.
2. **Gap cut** — among qualifiers (score-sorted), cut at the largest gap, but only when
   that gap is decisive (> 2× the mean gap in the window) — close-packed equals all come
   in, stragglers after a cliff don't.
3. **Bounds** — hard cap MAX_K; zero qualifiers is a feature: inject nothing and tell the
   agent "no confident matches — consider traversing the tree".
The table prints everything the mechanism sees so its choices can be judged by eye.

In [ ]:
Z_MIN = 3.5
MAX_K = 10


def adaptive_select(sims, use_gap_cut=True):
    mu, sigma = float(sims.mean()), float(sims.std())
    order = np.argsort(-sims)[:MAX_K]
    z = (sims[order] - mu) / sigma
    qualified = order[z >= Z_MIN]
    if use_gap_cut and len(qualified) > 1:
        s = sims[qualified]
        gaps = s[:-1] - s[1:]
        biggest = int(np.argmax(gaps))
        if gaps[biggest] > 2 * gaps.mean():   # only cut on a decisive cliff
            qualified = qualified[:biggest + 1]
    return list(qualified), mu, sigma


for prompt in PROMPTS:
    q = model.encode([prompt], normalize_embeddings=True)[0]
    sims = P @ q          # content-only: isolate the mechanism from the alpha question
    with_gap, mu, sigma = adaptive_select(sims, use_gap_cut=True)
    no_gap, _, _ = adaptive_select(sims, use_gap_cut=False)
    print("#" * 110)
    print(f"PROMPT: {prompt}")
    print(f"background: mu={mu:.3f} sigma={sigma:.3f}   floor(z>={Z_MIN}) = {mu + Z_MIN * sigma:.3f}")
    print(f"{'rank':>4} {'sim':>6} {'z':>5}  floor-only  +gap-cut  hit")
    for r, i in enumerate(np.argsort(-sims)[:MAX_K], 1):
        zz = (sims[i] - mu) / sigma
        m1 = "YES" if i in no_gap else "no "
        m2 = "YES" if i in with_gap else "no "
        print(f"{r:>4} {sims[i]:>6.3f} {zz:>5.1f}     {m1}        {m2}     {short(p_headers[i])[:95]}")
    verdict = lambda sel: f"inject {len(sel)}" if sel else "inject NOTHING -> traverse"
    diff = "  <-- DIFFER" if len(with_gap) != len(no_gap) else ""
    print(f"=> floor-only: {verdict(no_gap)}   |   +gap-cut: {verdict(with_gap)}{diff}")
    print()

## The combined recipe — everything that won, together

Final pipeline, per query:
1. **Vector leg** — α=0.5 path-blended similarity (`[√.5·content ; √.5·path]`), z-floor
   3.5 against its own full score distribution, cap 10.
2. **Keyword leg** — BM25 over FTS5(path + body), same z-floor logic over the matched-set
   score distribution, cap 10.
3. **Union of qualified candidates from both legs**, ranked by RRF position sum, cap 10.
   Empty union -> inject nothing + "consider traversing".
No gap cut (floor-only adaptive-k, per the comparison above). Rendered as a DIAGNOSTIC TABLE: for every candidate (union of both legs' top-10, so
excluded ones are visible too) — blended cosine + its z, BM25 + its z, per-leg qualified
flags, fused RRF score, and the final include verdict. Bodies omitted; this view is for
judging the mechanism, not the injection.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

ALPHA = 0.5
M_BLEND = BLENDS[ALPHA]
RRF_K = 60

for prompt in PROMPTS:
    q = model.encode([prompt], normalize_embeddings=True)[0]

    # vector leg: blended cosine over all paragraphs
    qa = np.concatenate([np.sqrt(1 - ALPHA) * q, np.sqrt(ALPHA) * q])
    sims = M_BLEND @ qa
    v_mu, v_sd = float(sims.mean()), float(sims.std())
    v_order = np.argsort(-sims)
    v_rank = [p_hashes[i] for i in v_order[:60]]
    sim_by_hash = {p_hashes[i]: float(sims[i]) for i in v_order[:200]}

    # keyword leg: BM25 (negated: higher = better), deduped to best row per hash
    terms = re.findall(r"[A-Za-z0-9_]+", prompt)
    rows = db.execute(
        "SELECT rowid, bm25(para_fts) FROM para_fts WHERE para_fts MATCH ?"
        " ORDER BY bm25(para_fts) LIMIT 2000", (" OR ".join(terms),)).fetchall()
    bm_by_hash = {}
    for rid, s in rows:
        bm_by_hash.setdefault(hash_by_id[rid], -s)
    bm_scores = np.array(list(bm_by_hash.values())) if bm_by_hash else np.array([0.0])
    k_mu, k_sd = float(bm_scores.mean()), float(bm_scores.std())
    k_rank = list(bm_by_hash)[:60]

    def z_v(h):
        return (sim_by_hash[h] - v_mu) / v_sd if h in sim_by_hash else None

    def z_k(h):
        return (bm_by_hash[h] - k_mu) / k_sd if h in bm_by_hash and k_sd > 0 else None

    v_qual = {h for h in v_rank[:MAX_K] if z_v(h) is not None and z_v(h) >= Z_MIN}
    k_qual = {h for h in k_rank[:MAX_K] if z_k(h) is not None and z_k(h) >= Z_MIN}

    def rrf(h):
        s = 0.0
        if h in v_rank:
            s += 1 / (RRF_K + v_rank.index(h) + 1)
        if h in k_rank:
            s += 1 / (RRF_K + k_rank.index(h) + 1)
        return s

    final = sorted(v_qual | k_qual, key=rrf, reverse=True)[:MAX_K]
    candidates = sorted(set(v_rank[:MAX_K]) | set(k_rank[:MAX_K]), key=rrf, reverse=True)

    rows_out = []
    for h in candidates:
        sv, zv, sk, zk = sim_by_hash.get(h), z_v(h), bm_by_hash.get(h), z_k(h)
        uri, _, sp = short(header_by_hash[h]).partition(" ")
        rows_out.append({
            "cosine": None if sv is None else round(sv, 3),
            "z_vec": None if zv is None else round(zv, 2),
            "vQ": "Y" if h in v_qual else "",
            "bm25": None if sk is None else round(sk, 1),
            "z_kw": None if zk is None else round(zk, 2),
            "kQ": "Y" if h in k_qual else "",
            "rrf": round(rrf(h), 4),
            "IN": "YES" if h in final else "",
            "file": uri.split("://", 1)[-1].split("@")[0].removeprefix("github.com/branow/"),
            "struct": sp,
        })
    df = pd.DataFrame(rows_out)
    verdict = (f"**INJECT {len(final)}**" if final
               else "**INJECT NOTHING** — no confident matches; consider traversing the tree")
    display(Markdown(
        f"### {prompt}\n"
        f"vector leg: mu={v_mu:.3f} sd={v_sd:.3f} floor={v_mu + Z_MIN * v_sd:.3f} · "
        f"keyword leg: mu={k_mu:.2f} sd={k_sd:.2f} floor={k_mu + Z_MIN * k_sd:.2f} · "
        f"Z_MIN={Z_MIN} → {verdict}"))
    display(df.style.apply(
        lambda r: ["background-color: #e8f5e9" if r["IN"] == "YES" else ""] * len(r), axis=1)
        .format(na_rep="—", precision=3).hide(axis="index"))

## Leg-weight sweep — how much BM25 do we actually need?

Instead of treating the legs as equals, weight the fusion: `score = w_vec/(60+rank_vec) +
w_kw/(60+rank_kw)`. Three configs: **vector-only** (w_kw=0 — is BM25 needed at all now that
paths are embedded?), **70/30**, and the current **50/50**. Floors (decided after the 13-prompt calibration rounds): Z_VEC=3.5 solo; keyword hits
need z_kw >= 6.0 AND weak vector support z_vec >= 3.0 — BM25 never qualifies alone (its
z-scores are largest exactly when most meaningless: rare legalese tokens). Strict on
purpose: a wrong injection costs tokens on every request; better to miss and traverse. Qualification stays per-leg (z ≥ Z_MIN); weights
change ranking and what survives the cap, not the qualified sets — so if keyword junk
merely sinks rather than disappears, the table will show it honestly.

Per prompt: candidates ordered by the 70/30 fusion, with each config's injection rank
(1 = injected first; empty = not injected under that config).

In [ ]:
Z_VEC = 3.5   # stricter: injection must be earned — cost of a wrong injection is paid on every request
Z_KW = 6.0        # BM25 held to a much higher bar (6-7 range; 6 keeps the weakest real kw win at 7.55)
Z_KW_SUPPORT = 3.0  # ...and NEVER alone: a keyword hit needs at least weak vector support
WEIGHTS = {"vec-only": (1.0, 0.0), "70/30": (0.7, 0.3), "50/50": (0.5, 0.5)}

for prompt in PROMPTS:
    q = model.encode([prompt], normalize_embeddings=True)[0]
    qa = np.concatenate([np.sqrt(1 - ALPHA) * q, np.sqrt(ALPHA) * q])
    sims = M_BLEND @ qa
    v_mu, v_sd = float(sims.mean()), float(sims.std())
    v_order = np.argsort(-sims)
    v_rank = [p_hashes[i] for i in v_order[:60]]
    sim_by_hash = {p_hashes[i]: float(sims[i]) for i in v_order[:200]}

    terms = re.findall(r"[A-Za-z0-9_]+", prompt)
    rows = db.execute(
        "SELECT rowid, bm25(para_fts) FROM para_fts WHERE para_fts MATCH ?"
        " ORDER BY bm25(para_fts) LIMIT 2000", (" OR ".join(terms),)).fetchall()
    bm_by_hash = {}
    for rid, s in rows:
        bm_by_hash.setdefault(hash_by_id[rid], -s)
    bm_scores = np.array(list(bm_by_hash.values())) if bm_by_hash else np.array([0.0])
    k_mu, k_sd = float(bm_scores.mean()), float(bm_scores.std())
    k_rank = list(bm_by_hash)[:60]

    z_v = lambda h: (sim_by_hash[h] - v_mu) / v_sd if h in sim_by_hash else None
    z_k = lambda h: (bm_by_hash[h] - k_mu) / k_sd if h in bm_by_hash and k_sd > 0 else None
    v_qual = {h for h in v_rank[:MAX_K] if z_v(h) is not None and z_v(h) >= Z_VEC}
    k_qual = {h for h in k_rank[:MAX_K]
              if z_k(h) is not None and z_k(h) >= Z_KW
              and z_v(h) is not None and z_v(h) >= Z_KW_SUPPORT}

    def fused(h, wv, wk):
        s = 0.0
        if wv and h in v_rank:
            s += wv / (RRF_K + v_rank.index(h) + 1)
        if wk and h in k_rank:
            s += wk / (RRF_K + k_rank.index(h) + 1)
        return s

    injected = {}
    for name, (wv, wk) in WEIGHTS.items():
        pool = v_qual if wk == 0 else (v_qual | k_qual)
        ranked = sorted(pool, key=lambda h: fused(h, wv, wk), reverse=True)[:MAX_K]
        injected[name] = {h: r + 1 for r, h in enumerate(ranked)}

    candidates = sorted(set(v_rank[:MAX_K]) | set(k_rank[:MAX_K]),
                        key=lambda h: fused(h, 0.7, 0.3), reverse=True)
    rows_out = []
    for h in candidates:
        sv, zv, sk, zk = sim_by_hash.get(h), z_v(h), bm_by_hash.get(h), z_k(h)
        uri, _, sp = short(header_by_hash[h]).partition(" ")
        rows_out.append({
            "cosine": None if sv is None else round(sv, 3),
            "z_vec": None if zv is None else round(zv, 2),
            "bm25": None if sk is None else round(sk, 1),
            "z_kw": None if zk is None else round(zk, 2),
            "vec-only": injected["vec-only"].get(h, ""),
            "70/30": injected["70/30"].get(h, ""),
            "50/50": injected["50/50"].get(h, ""),
            "file": uri.split("://", 1)[-1].split("@")[0].removeprefix("github.com/branow/"),
            "struct": sp,
        })
    df = pd.DataFrame(rows_out)
    counts = " · ".join(f"{n}: inject {len(v)}" if v else f"{n}: NOTHING -> traverse"
                        for n, v in injected.items())
    display(Markdown(f"### {prompt}\n{counts}"))
    display(df.style.apply(
        lambda r: ["background-color: #e8f5e9" if r["70/30"] != "" else ""] * len(r), axis=1)
        .format(na_rep="—", precision=3).hide(axis="index"))

## Observations — FROZEN RECIPE (2026-07-25)

**The retrieval recipe:** α=0.5 path-blended vectors (granite-97m) + BM25 over FTS5(path+body),
qualification Z_VEC ≥ 3.5 solo / keyword z_kw ≥ 6.0 AND vector support z_vec ≥ 3.0 (BM25 never
alone), fusion = weighted RRF 70/30 (vector/keyword), cap 10, empty set → "no confident
matches — traverse the tree".

**How it got here (13 prompts: 10 answerable + 3 no-answer controls):**
1. Content-only top-k: 1 clean win, boilerplate poisoning (imports) + vocabulary mismatch
   (license/Ukrainian/gradle) — 3 hard failures.
2. Hybrid RRF 50/50 fixed exact-token failures (gradle #1) but BM25's z-floor is degenerate:
   its background is stopword matches, so legalese scores z=5-9 while meaning nothing.
3. α=0.5 path blend fixed Ukrainian/license/photo (path carries what bodies don't say);
   struct paths deliberately not embedded/indexed (names already in bodies, rest is noise).
4. Adaptive-k: gap-cut dropped (over-aggressive, amputated useful borderline context);
   floor-only kept.
5. Calibration on 13 samples killed single-threshold answerability detection: junk controls
   reach z_vec 3.56 while real answers start at 3.26 — the bands OVERLAP. Embedding score
   statistics measure relative outlierness, not answerability.
6. Final strict policy chosen deliberately (injection costs tokens on every request):
   high precision, misses fail SAFE to traversal.

**Scorecard at frozen settings:** controls 2/3 correctly empty; real prompts 8/10 inject
(lean, right file #1), 2/10 fall to traverse (uk-translation z=3.44, database z=3.26 —
below the 3.5 floor).

**Known residuals (accepted):** (a) Helm control injects 1 (keycloak.ts z=3.56) — no
threshold separates it; the fixes are an LLM relevance gate or a bigger embedding model
(both deferred). (b) The two strict-policy misses above. (c) Constants calibrated on 13
prompts / 4.2k paragraphs — re-validate at scale and on the final-run model (Qwen3).
(d) mvnw/gradlew wrapper boilerplate still in corpus — filter.toml gap, see TODO.